In [1]:
import random

import torch

In [2]:
g = torch.Generator().manual_seed(123456)

w = torch.tensor([3.0, 4.5], dtype=torch.float32)
b = 10.0
noise = 0.01
num_train, num_val = 1200, 800

n = num_train + num_val
d = len(w)

X = torch.randn(n, d, generator=g)
w = w.reshape((-1, 1))
noise = torch.randn(n, 1, generator=g) * noise


y = (X @ w) + b + noise

X.shape, w.shape, y.shape

(torch.Size([2000, 2]), torch.Size([2, 1]), torch.Size([2000, 1]))

In [3]:
X[0, :], y[0]

(tensor([-1.1218, -1.8445]), tensor([-1.6529]))

In [4]:
train = False
batch_size = 32

if train:
    indices = list(range(0, num_train))
else:
    indices = list(range(num_train, n))

dataset = torch.utils.data.TensorDataset(X[indices], y[indices])
dataloader = torch.utils.data.DataLoader(dataset, batch_size, shuffle=train)

len(dataloader)

25

In [6]:
X, y = next(iter(dataloader))
X.shape, y.shape

(torch.Size([32, 2]), torch.Size([32, 1]))

In [25]:
class SyntheticRegressionData:
    def __init__(self, w, b, noise, num_train, num_val, batch_size):
        self.w = w
        self.b = b
        self.noise = noise
        self.num_train = num_train
        self.num_val = num_val
        self.batch_size = batch_size
        
        self.n = num_train + num_val
        self.d = len(w)

        self.X = torch.randn(self.n, self.d)
        e = torch.randn(self.n, 1) * self.noise

        self.y = (self.X @ self.w.reshape((-1, 1))) + self.b + e

    def get_dataloader(self, train):
        indices = slice(0, self.num_train) if train else slice(num_train, self.n)
        
        return self._get_tensorloader(train, indices=indices)

    def _get_tensorloader(self, train, indices=slice(0, None)):
        dataset = torch.utils.data.TensorDataset(self.X[indices], self.y[indices])
        
        return torch.utils.data.DataLoader(dataset, self.batch_size, shuffle=train)

In [26]:
w = torch.tensor([2, -3.4])
b = 4.2

num_train = 1200
num_val = 800
batch_size = 32

data = SyntheticRegressionData(w, b, 0.01, num_train, num_val, batch_size)

In [27]:
loader = data.get_dataloader(train=True)

In [28]:
X, y = next(iter(loader))

X.shape, y.shape

(torch.Size([32, 2]), torch.Size([32, 1]))

In [34]:
class SGD:
    def __init__(self, params, lr):
        self.params = params
        self.lr = lr

    def step(self):
        for p in params:
            p -= self.lr * p.grad

    def zero_grad(self):
        for p in params:
            if p.grad is not None:
                p.grad.zero_()

In [33]:
class LinearRegressionScratch:
    def __init__(self, num_inputs, lr, sigma=0.01):
        self.num_inputs = num_inputs
        self.lr = lr
        self.sigma = sigma
        
        self.w = torch.normal(0, sigma, (num_inputs, 1), requires_grad=True)
        self.b = torch.zeros(1, requires_grad=True)

    def forward(self, X):
        return (X @ self.w) + self.b

    def loss(self, y_hat, y):
        l = ((y_hat - y) ** 2) / 2
        return l.mean()

    def configure_optimizer(self):
        return SGD([self.w, self.b], self.lr)